In [19]:
# !pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [1]:
import os
import re
import base64
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from configparser import ConfigParser
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from collections import defaultdict

# OAuth Scopes
SCOPES = ['https://www.googleapis.com/auth/gmail.compose']

def load_config():
    """Load configuration from INI file with validation"""
    config = ConfigParser()
    config.optionxform = str  # Preserve case sensitivity
    config.read('config.ini', encoding='utf-8')  # Explicit encoding
    
    if not config.read('config.ini'):
        raise ValueError("Could not read config.ini file")
    
    # Verify required sections
    required_sections = ['CLIENTS', 'EMAIL', 'SETTINGS']
    for section in required_sections:
        if not config.has_section(section):
            raise ValueError(f"Missing section [{section}] in config.ini")
    
    return config

def authenticate_gmail():
    """More robust authentication with error handling"""
    creds = None
    token_file = 'token.json'
    
    # 1. Check for existing token
    if os.path.exists(token_file):
        try:
            creds = Credentials.from_authorized_user_file(token_file, SCOPES)
            
            # Check if token needs refresh
            if creds and creds.expired and creds.refresh_token:
                print("Refreshing access token...")
                creds.refresh(Request())
                # Save refreshed token
                with open(token_file, 'w') as token:
                    token.write(creds.to_json())
                return build('gmail', 'v1', credentials=creds)
                
            elif creds and creds.valid:
                return build('gmail', 'v1', credentials=creds)
                
        except Exception as e:
            print(f"Token load/refresh failed: {e}")
            os.remove(token_file)
    
    # 2. Fresh authentication flow
    print("Starting new authentication flow...")
    try:
        flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
        creds = flow.run_local_server(port=0, timeout_seconds=60)
        
        # Save the new token
        with open(token_file, 'w') as token:
            token.write(creds.to_json())
            
        return build('gmail', 'v1', credentials=creds)
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        raise

def parse_transactions(file_path, client_names):
    """Parse transaction instructions from file"""
    transactions = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                # Match the full line format
                # e.g. For Client ID: H20405 Name: Madhavi Kulaye, Please SELL 101 Shares of Anup Engineering
                match = re.match(
                    r'.*Client ID:\s+(\S+)\s+Name:\s+([^,]+),\s+Please\s+(\w+)\s+(\d*)\s+Shares of\s+(.+)',
                    line,
                    re.IGNORECASE  # <-- fixes "Shares" vs "shares" mismatch
                )

                if match:
                    client_code  = match.group(1).strip()
                    client_name  = match.group(2).strip()
                    action       = match.group(3).strip()
                    quantity     = match.group(4).strip()  # empty string if blank
                    stock        = match.group(5).strip()

                    transactions.append({
                        'client_code': client_code,
                        'client_name': client_names.get(client_code, client_name),
                        'action':      action,
                        'quantity':    quantity if quantity else 'ALL',  # handle blank qty
                        'stock':       stock
                    })
                else:
                    print(f"No match for line: {line}")  # helpful debug output

    except Exception as e:
        print(f"Error reading transactions: {str(e)}")

    return transactions

def get_email_body_grouped(config, client_code, client_name, transactions):
    """Format email body with multiple transactions for one client"""
    raw_template = config['EMAIL']['approval_body']
    
    # Handle both literal \n and actual newlines
    if '\\n' in raw_template:
        template = raw_template.replace('\\n', '\n')
    else:
        template = raw_template

    # Build transaction lines
    transaction_lines = ""
    for tx in transactions:
        transaction_lines += f"  • {tx['action'].capitalize()} {tx['quantity']} Shares of {tx['stock']}\n"

    return template.format(
        client_code=client_code,
        client_names=client_name,
        transactions=transaction_lines,
        # Keep backward compat in case template uses old fields
        action=transactions[0]['action'].capitalize() + 'ing',
        quantity=transactions[0]['quantity'],
        stock=transactions[0]['stock']
    )
    
def create_grouped_draft(service, config, client_code, client_name, transactions):
    """Create a single draft email per client covering all their transactions"""
    try:
        to_email = config['CLIENTS'].get(client_code)

        if not to_email:
            print(f"No email found for client {client_code} ({client_name})")
            return

        msg = MIMEMultipart()
        msg['Subject'] = config['EMAIL']['approval_subject'].format(client_code=client_code)
        msg['From']    = config['EMAIL']['sender']
        msg['To']      = to_email.strip()

        if config['EMAIL'].get('carbon_copy'):
            msg['Cc'] = config['EMAIL']['carbon_copy']

        body = get_email_body_grouped(config, client_code, client_name, transactions)
        msg.attach(MIMEText(body, 'plain', 'utf-8'))

        raw_message  = base64.urlsafe_b64encode(msg.as_bytes()).decode()
        draft        = {'message': {'raw': raw_message}}
        created_draft = service.users().drafts().create(
            userId='me',
            body=draft
        ).execute()
        print(f"Draft created for {client_code} — {client_name} "
              f"({len(transactions)} transaction(s))")

    except Exception as error:
        print(f"Error creating draft for {client_code}: {error}")


def main():
    try:
        # Load configuration
        config = load_config()
        print("Configuration loaded successfully!")

        # Load client names
        client_names = dict(config['CLIENT_NAMES']) if config.has_section('CLIENT_NAMES') else {}

        # Get transaction file path
        transactions_file = config['SETTINGS']['transactions_file']
        print(transactions_file)
        if not os.path.exists(transactions_file):
            raise FileNotFoundError(f"Transaction file not found: {transactions_file}")

        # Parse transactions
        transactions = parse_transactions(transactions_file, client_names)
        if not transactions:
            print("No valid transactions found")
            return

        # ── Group all transactions by client code ──────────────────────────
        grouped = defaultdict(list)
        for tx in transactions:
            grouped[tx['client_code']].append(tx)
        # ───────────────────────────────────────────────────────────────────

        # Authenticate with Gmail
        try:
            service = authenticate_gmail()
        except Exception as e:
            print(f"Fatal authentication error: {e}")
            raise

        # Create ONE draft per client covering ALL their transactions
        for client_code, txs in grouped.items():
            client_name = txs[0]['client_name']
            create_grouped_draft(service, config, client_code, client_name, txs)

    except Exception as e:
        print(f"An error occurred: {str(e)}")


if __name__ == '__main__':
    main()

Configuration loaded successfully!
transactions.txt
Refreshing access token...
Draft created for H20405 — Madhavi Kulaye (2 transaction(s))
Draft created for H20404 — Rohini Kulaye (2 transaction(s))
Draft created for H20427 — Yatin Gawde (2 transaction(s))
Draft created for H20431 — Pramod Pawar (2 transaction(s))
Draft created for H20467 — Rahul/Kirti Vaknalli (2 transaction(s))
Draft created for H20468 — Prashant Parab (2 transaction(s))
Draft created for H21398 — Nikhil Phatak (2 transaction(s))
Draft created for H22636 — Parita/Paresh Padave (2 transaction(s))
Draft created for H22295 — Snehal Pailkar (2 transaction(s))
Draft created for H1837 — Jayshree/Hemant Chaudhari (2 transaction(s))
Draft created for H22951 — Nikhil Parab (2 transaction(s))
Draft created for H2031 — Anurag Pawar (2 transaction(s))
Draft created for H0822 — Shubham Pawar (2 transaction(s))
Draft created for H3665 — Ashika/Ashish Surve (2 transaction(s))
Draft created for H3733 — Amol Modak (2 transaction(s))